# Thursday Assignment Exercises

In [1]:
import sys
sys.path.append('.')
import pandas as pd
import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

from src.data_prep import prepare_stock_sequences, clean_chat_logs, extract_churn_features
from src.models_stock import StockLSTM, train_stock_lstm, evaluate_stock_lstm, AutoregressiveBaseline, evaluate_autoregressive
from src.models_churn import TabularChurnModel, evaluate_churn_cost_model, get_risk_list
from src.bptt import run_vanishing_gradient_demo


## Sub-step 1: Stock Data Preparation
**Window Size Chosen:** 20 days. This represents approximately one month of trading days, allowing the model to capture short-term momentum and patterns.
**Split Strategy:** Temporal split (first 80% chronologically for training, last 20% for testing).
**Why random split is unacceptable:** Time-series data has temporal dependency (autocorrelation). A random split would leak future information into the training set, causing the model to overfit and artificially inflating performance while failing in production.


In [2]:
df_stock = pd.read_csv('data/stock_prices.csv')
ticker = df_stock['ticker'].unique()[0]

X_train, X_test, y_train, y_test, scaler, df_tkr = prepare_stock_sequences(
    df_stock, ticker, window_size=20, train_ratio=0.8
)
print(f"Prepared {(X_train.shape, y_train.shape)} training samples for {ticker}.")


Prepared ((584, 20, 1), (584, 1)) training samples for RELIANCE.


## Sub-step 2: Chat Logs Timestamp Cleaning & EDA

In [3]:
df_chat = pd.read_csv('data/chat_logs.csv')
df_chat_clean = clean_chat_logs(df_chat)
print(f"Cleaned chat logs shape: {df_chat_clean.shape}")

df_tabular = extract_churn_features(df_chat_clean)
print(f"Extracted Tabular Feature shape: {df_tabular.shape}")
print(df_tabular.head(3))


Cleaned chat logs shape: (2500, 10)
Extracted Tabular Feature shape: (2500, 17)
        chat_id  duration_min  num_turns  tenure_months  churned_30d  \
1087  CHT001087             6         11             58            1   
1427  CHT001427            12          2             33            0   
739   CHT000739             6          5              1            0   

      customer_sentiment_confused  customer_sentiment_frustrated  \
1087                        False                          False   
1427                        False                          False   
739                         False                          False   

      customer_sentiment_neutral  customer_sentiment_satisfied  \
1087                       False                          True   
1427                       False                          True   
739                         True                         False   

      primary_intent_cancellation_request  primary_intent_general_enquiry  \
1087            

## Sub-step 3: LSTM Stock Prediction

In [4]:
# Using MAE directly on original scale as evaluation metric for trading desk.
# A model needs an MAE low enough that the predicted variance doesn't overshadow trading fees and slippage to be profitable.

lstm_model = StockLSTM(hidden_dim=32, num_layers=1)
train_stock_lstm(lstm_model, X_train, y_train, epochs=25, lr=0.005)

mae_lstm, preds_lstm, y_test_orig = evaluate_stock_lstm(lstm_model, X_test, y_test, scaler)
print(f"LSTM Test MAE: {mae_lstm:.4f}")


LSTM Test MAE: 456.6303


## Sub-step 4 & 5: Churn Model and Cost Optimization
Hypothesis: A tabular model capturing max/mean aggregates per customer suffices instead of a complex sequence model, as churn indicators are often holistic limits (e.g. low sentiment over time).

In [5]:
X_c = df_tabular.drop(columns=['chat_id', 'churned_30d']).fillna(0).values
y_c = df_tabular['churned_30d'].values

# Split
idx = int(len(X_c) * 0.8)
X_c_train, X_c_test = X_c[:idx], X_c[idx:]
y_c_train, y_c_test = y_c[:idx], y_c[idx:]
cid_test = df_tabular['chat_id'].iloc[idx:].values

churn_model = TabularChurnModel()
churn_model.fit(X_c_train, y_c_train)
y_probs = churn_model.predict_proba(X_c_test)

cost_eval = evaluate_churn_cost_model(y_c_test, y_probs, cost_tp=50, cost_fp=15, cost_fn=200)
print(f"Cost Model Results: {cost_eval}")

risk_list = get_risk_list(cid_test, y_probs, cost_eval['best_threshold'])
print("Top 5 Customers to Target:")
print(risk_list.head())


Cost Model Results: {'best_threshold': np.float64(0.060000000000000005), 'min_cost': np.int64(10180), 'num_contacted_per_month': np.int64(350), 'roc_auc': 0.6927472046802811}
Top 5 Customers to Target:
    customer_id  churn_prob
430   CHT000129        0.87
85    CHT001023        0.75
254   CHT001533        0.73
194   CHT000433        0.69
16    CHT000373        0.69


## Sub-step 6 (Hard): Autoregressive Baseline
Comparing simple Linear Regression across flattened windows vs LSTM.

In [6]:
ar_model = AutoregressiveBaseline()
ar_model.fit(X_train, y_train)
mae_ar, preds_ar, _ = evaluate_autoregressive(ar_model, X_test, y_test, scaler)
print(f"Autoregressive Baseline Test MAE: {mae_ar:.4f}")
print(f"LSTM MAE: {mae_lstm:.4f}")
print("Diagnosis: If AR beats LSTM, sequence momentum is straightforward and deep patterns are minimal. \nIf LSTM wins, it captured non-linear cyclical sequences that linear combination filters missed.")


Autoregressive Baseline Test MAE: 24.0152
LSTM MAE: 456.6303
Diagnosis: If AR beats LSTM, sequence momentum is straightforward and deep patterns are minimal. 
If LSTM wins, it captured non-linear cyclical sequences that linear combination filters missed.


## Sub-step 7 (Hard): BPTT Vanishing Gradients

In [7]:
norms_5, norms_50 = run_vanishing_gradient_demo()

print(f"Max Gradient (Length 5): {np.max(norms_5):.8f}")
print(f"Min Gradient (Length 5): {np.min(norms_5):.8f}")

print(f"Max Gradient (Length 50): {np.max(norms_50):.8f}")
print(f"Min Gradient (Length 50): {np.min(norms_50):.8f}")
print("\nAs sequence length increases, raw gradients shrink exponentially through the backward graph into near-zero territory early in the sequence, hence LSTMs use additive cell states.")


Max Gradient (Length 5): 1.78805388
Min Gradient (Length 5): 0.00413678
Max Gradient (Length 50): 2.12508742
Min Gradient (Length 50): 0.00000000

As sequence length increases, raw gradients shrink exponentially through the backward graph into near-zero territory early in the sequence, hence LSTMs use additive cell states.
